# How the exact probabilities are computed

Every unopened cell carries a number: the fraction of consistent mine layouts in
which it holds a mine. That is a definition, not an estimate — count the layouts,
count the ones with a mine there, divide.

The difficulty is that on an Expert board there are about $10^{82}$ ways to place
the mines. This notebook walks the pipeline that makes the count tractable, running
each stage on real positions, and finishes by checking the whole thing against brute
force.

The prose version is [`constraint-estimator.md`](constraint-estimator.md). The engine
itself is Rust, in `minesweeper_core/src/probability/`; `minesweeper_ref.py` beside
this notebook is a readable Python twin that follows it step for step.

**Requires numpy and nothing else.**

In [1]:
import itertools, math, random, sys
sys.path.insert(0, '.')
import minesweeper_ref as ms

print('reference module loaded')

reference module loaded


## A position to work on

`from_text` builds a board from a picture of it. `*` is a mine, `.` an unopened
empty square, a digit or space an opened one. The numbers shown are *computed* from
the mine positions, so the picture cannot contradict itself.

Below, `▒` is unopened and `·` is an opened zero. The mines are of course invisible
to the solver — we only know where they are because we are writing the example.

In [2]:
# `*` is a mine, `.` an unopened empty square, `o` an opened one. The numbers
# are computed from the mine positions, so the picture cannot contradict itself.
board = ms.from_text("""
...o**ooo
...*..oo.
......*oo
.......oo
*...*..oo
....*...*
""")

print(board.render())
print()
print(f'{len(board.hidden_cells())} unopened cells, {board.mines_count} mines')

    ▒    ▒    ▒    2    ▒    ▒    1    ·    ·
    ▒    ▒    ▒    ▒    ▒    ▒    2    1    ▒
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    1    ·
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    1    ·
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    1    1
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    ▒    ▒

42 unopened cells, 8 mines


## 1. Constraints

Each visible number becomes *"exactly k of these unopened neighbours are mines"*.
Plus one global constraint: the board holds a fixed number of mines in total.

A flagged cell stays in as an unknown — a flag is the player's opinion, and the
solver has no reason to believe it. A visible `0` is a constraint too: "none of my
neighbours is a mine" is as strong a statement as any other number.

In [3]:
hidden, constraints, mines_to_place = ms.build_constraints(board)

print(f'{len(hidden)} unknowns, {len(constraints)} constraints, {mines_to_place} mines to place')
print()
for cells, required in constraints[:8]:
    squares = ' '.join(str(hidden[i]) for i in cells)
    print(f'exactly {required} mine(s) among: {squares}')
if len(constraints) > 8:
    print(f'... and {len(constraints) - 8} more')

42 unknowns, 11 constraints, 8 mines to place

exactly 2 mine(s) among: (2, 0) (4, 0) (2, 1) (3, 1) (4, 1)
exactly 1 mine(s) among: (5, 0) (5, 1)
exactly 0 mine(s) among: (8, 1)
exactly 0 mine(s) among: (8, 1)
exactly 2 mine(s) among: (5, 0) (5, 1) (5, 2) (6, 2)
exactly 1 mine(s) among: (8, 1) (6, 2)
exactly 1 mine(s) among: (8, 1) (6, 2) (6, 3)
exactly 0 mine(s) among: (8, 1)
... and 3 more


## 2. Propagation — what local rules alone settle

Two rules, applied until nothing changes:

- a constraint whose mines are all accounted for makes its remaining cells **safe**;
- a constraint with exactly as many undecided cells as mines left makes them all **mines**.

The board's total mine count is the same pair of rules applied globally.

This costs no search. It is **sound but incomplete** — it never marks a cell wrongly,
but it misses cells only a full enumeration can prove. A caller must never read
*"not proven"* as *"not certain"*. `probability::certain_cells` exposes exactly this,
and auto-play iterates on it, paying for a full solve only once it runs dry.

In [4]:
proven_mines, proven_safe, reduced, mines_left = ms.propagate(
    len(hidden), constraints, mines_to_place)

print(f'proven mines : {[hidden[i] for i in proven_mines]}')
print(f'proven safe  : {[hidden[i] for i in proven_safe]}')
print(f'still open   : {len(hidden) - len(proven_mines) - len(proven_safe)} cells, '
      f'{mines_left} mines, {len(reduced)} constraints left')

proven mines : [(6, 2)]
proven safe  : [(8, 1), (6, 3), (6, 4)]
still open   : 38 cells, 7 mines, 5 constraints left


## 3. Decomposition — the stage that matters

Two cells are related only if some number counts them both, directly or through a
chain of other cells. So the border falls into groups that share no cell at all.
Union-find over *"appears in the same constraint"* finds them.

The groups are independent **apart from the board's total mine count**. That is the
whole trick, and it turns a product into a sum:

```
one problem:     solutions(A) × solutions(B) × solutions(C)
three problems:  solutions(A) + solutions(B) + solutions(C)
```

In [5]:
decided = set(proven_mines) | set(proven_safe)
free = [i for i in range(len(hidden)) if i not in decided]
remap = {c: i for i, c in enumerate(free)}
reduced_local = [([remap[c] for c in cs], r) for cs, r in reduced]

components = ms.decompose(reduced_local, len(free))
in_a_component = {c for cells, _ in components for c in cells}
interior = [i for i in range(len(free)) if i not in in_a_component]

print(f'{len(components)} independent groups, plus {len(interior)} interior cells')
for n, (cells, cons) in enumerate(components):
    squares = ' '.join(str(hidden[free[c]]) for c in cells)
    print(f'  group {n}: {len(cells)} cells, {len(cons)} constraints  →  {squares}')

product = math.prod(2 ** len(cells) for cells, _ in components)
total = sum(2 ** len(cells) for cells, _ in components)
print()
print(f'as one problem : {product:,} assignments to walk')
print(f'group by group : {total:,}')

3 independent groups, plus 27 interior cells
  group 0: 5 cells, 1 constraints  →  (2, 0) (4, 0) (2, 1) (3, 1) (4, 1)
  group 1: 3 cells, 2 constraints  →  (5, 0) (5, 1) (5, 2)
  group 2: 3 cells, 2 constraints  →  (6, 5) (7, 5) (8, 5)

as one problem : 2,048 assignments to walk
group by group : 48


## 4. What a group returns — and why it is not a probability

How likely a group's layouts are depends on how many mines the *rest* of the board
takes. So a probability is not a property of the group.

What *is* a property of the group — independent of the total mine count and of every
other group — is the count of layouts per mine count:

- `ways[k]` — layouts of this group using exactly `k` mines
- `cell_ways[c][k]` — how many of those have cell `c` as a mine

This form is why groups can be combined at all. It is also why they can be **cached
across moves**: the solution is a function of exactly the group's squares and the
numbers around them, so an entry can never go stale — a board that changes a group
changes its cache key too. Measured reuse after one reveal: 97% of groups on
50×50/150.

In [6]:
solutions = [ms.solve_component(cells, cons) for cells, cons in components]

for n, ((cells, _), (ways, cell_ways)) in enumerate(zip(components, solutions)):
    print(f'group {n}:  ways[k] = {ways}   ({sum(ways)} layouts in total)')
    for local, cell in enumerate(cells):
        print(f'    {str(hidden[free[cell]]):8} is a mine in {cell_ways[local]}')
    print()

group 0:  ways[k] = [0, 0, 10, 0, 0, 0]   (10 layouts in total)
    (2, 0)   is a mine in [0, 0, 4, 0, 0, 0]
    (4, 0)   is a mine in [0, 0, 4, 0, 0, 0]
    (2, 1)   is a mine in [0, 0, 4, 0, 0, 0]
    (3, 1)   is a mine in [0, 0, 4, 0, 0, 0]
    (4, 1)   is a mine in [0, 0, 4, 0, 0, 0]

group 1:  ways[k] = [0, 2, 0, 0]   (2 layouts in total)
    (5, 0)   is a mine in [0, 1, 0, 0]
    (5, 1)   is a mine in [0, 1, 0, 0]
    (5, 2)   is a mine in [0, 0, 0, 0]

group 2:  ways[k] = [0, 2, 0, 0]   (2 layouts in total)
    (6, 5)   is a mine in [0, 0, 0, 0]
    (7, 5)   is a mine in [0, 1, 0, 0]
    (8, 5)   is a mine in [0, 1, 0, 0]



## 5. Combining

The number of whole-board layouts using `b` border mines is the convolution of the
groups' `ways`. The **interior** — cells no number speaks about — takes whatever is
left, in `C(interior, left)` ways.

For each group the code needs *everything except this group*. It gets that from a
prefix and a suffix convolution, `prefix[j] × suffix[j+1]`, rather than by dividing
the total out — division breaks wherever a group contributes a zero.

Every interior cell gets the same number as every other, since nothing distinguishes
them.

In [7]:
probs = ms.exact_probabilities(board)

print(board.render(probs))
print()
print(f'these sum to {sum(probs.values()):.6f}; there are {board.mines_count} mines')
print()
still_hidden = sorted(c for c in board.mines if c in probs)
print('where the mines really are: ' + ', '.join(f'{c}={probs[c]:.0%}' for c in still_hidden))

  11%  11%  40%    2  40%  50%    1    ·    ·
  11%  11%  40%  40%  40%  50%    2    1   0%
  11%  11%  11%  11%  11%   0% 100%    1    ·
  11%  11%  11%  11%  11%  11%   0%    1    ·
  11%  11%  11%  11%  11%  11%   0%    1    1
  11%  11%  11%  11%  11%  11%   0%  50%  50%

these sum to 8.000000; there are 8 mines

where the mines really are: (0, 4)=11%, (3, 1)=40%, (4, 0)=40%, (4, 4)=11%, (4, 5)=11%, (5, 0)=50%, (6, 2)=100%, (8, 5)=50%


The sum is the cheapest sanity check there is, and it catches a mis-scaled weight
table, a wrong interior window and an off-by-one in the prefix/suffix chain alike.
The Rust asserts it on every solve in debug builds.

## 6. Checking it against brute force

The pipeline is only worth having if it agrees with counting every layout directly.
Brute force is hopeless past about 20 unopened cells — which is the whole reason the
pipeline exists — but that is enough to test on.

In [8]:
random.seed(11)
worst, checked = 0.0, 0

for _ in range(400):
    w, h = random.choice([(5, 4), (6, 4), (5, 5), (6, 5)])
    squares = [(x, y) for y in range(h) for x in range(w)]
    mines = set(random.sample(squares, random.randint(3, 7)))
    b = ms.Board(w, h, mines)
    b.reveal(*random.choice([c for c in squares if c not in mines]))
    if not 1 <= len(b.hidden_cells()) <= 20:
        continue
    oracle = ms.brute_force(b)
    if not oracle:
        continue
    exact = ms.exact_probabilities(b)
    checked += 1
    worst = max(worst, max(abs(exact[c] - oracle[c]) for c in oracle))

print(f'{checked} random positions, worst disagreement {worst:.2e}')

181 random positions, worst disagreement 0.00e+00


## 7. The first trap: rescaled weights

`C(n, k)` overflows `f64` on any real board, so the weights are computed in log space
and rescaled by subtracting a peak. The scale cancels because the *same table*
divides numerator and denominator — which is also why you must **never compare
weights across two tables**.

The first version subtracted the table's **global** peak. On a sparse board the
reachable range of `k` is far out in the tail, so everything underflowed to zero, the
total came out zero, and the whole grid read **0%** — which every front-end treats as
proof that a cell is safe, and opens. A 30-game sweep caught it as two detonations on
50×50/150.

In [9]:
import numpy as np

n, k_min, k_max = 2500, 140, 150          # interior cells, reachable mines
ln_fact = np.cumsum([0.0] + [math.log(i) for i in range(1, n + 1)])
ln_w = np.array([ln_fact[n] - ln_fact[k] - ln_fact[n - k] for k in range(n + 1)])

print(f'C({n}, {k_max}) as a float      : {math.exp(ln_w[k_max]):.3g}  (f64 max is {sys.float_info.max:.3g})')
print()
global_peak = ln_w.max()                   # the bug
reachable_peak = ln_w[k_min:k_max + 1].max()  # the fix
print(f'global peak    : ln = {global_peak:8.1f}  at k = {int(ln_w.argmax())}')
print(f'reachable peak : ln = {reachable_peak:8.1f}  over k = {k_min}..{k_max}')
print()
for k in (k_min, k_max):
    print(f'  k={k}:  scaled by global peak    = {math.exp(ln_w[k] - global_peak):.3g}')
    print(f'         scaled by reachable peak = {math.exp(ln_w[k] - reachable_peak):.3g}')

bad = np.exp(ln_w[k_min:k_max + 1] - global_peak)
print()
print(f'total weight with the global peak    : {bad.sum():.3g}   <- every cell reads 0%')
print(f'total weight with the reachable peak : {np.exp(ln_w[k_min:k_max+1] - reachable_peak).sum():.3g}')

C(2500, 150) as a float      : 8.97e+244  (f64 max is 1.8e+308)

global peak    : ln =   1728.7  at k = 1250
reachable peak : ln =    564.0  over k = 140..150

  k=140:  scaled by global peak    = 0
         scaled by reachable peak = 8.07e-13
  k=150:  scaled by global peak    = 0
         scaled by reachable peak = 1

total weight with the global peak    : 0   <- every cell reads 0%
total weight with the reachable peak : 1.07


## 8. The second trap: a partial search is not a small answer

If a group exhausts its node budget, **the whole answer is discarded** — not just
that group's.

A depth-first walk that stops early has covered a lexicographic *prefix* of the
search space. A cell whose subtree was never reached is a mine in zero of the layouts
seen, which reads as 0%, which every caller treats as proof. A truncated walk does
not give a rough answer; it gives a confident wrong one.

Below: enumerate one group honestly, then stop a quarter of the way in and compare.

In [10]:
cells, cons = max(components, key=lambda c: len(c[0]))
size = len(cells)

def enumerate_group(limit=None):
    ways, cell_ways, seen = [0] * (size + 1), [[0] * (size + 1) for _ in range(size)], 0
    for bits in itertools.product((0, 1), repeat=size):   # lexicographic, as DFS is
        if limit is not None and seen >= limit:
            break
        seen += 1
        if any(sum(bits[i] for i in cs) != r for cs, r in cons):
            continue
        ways[sum(bits)] += 1
        for c, bit in enumerate(bits):
            if bit:
                cell_ways[c][sum(bits)] += 1
    return ways, cell_ways

full_ways, full_cell = enumerate_group()
part_ways, part_cell = enumerate_group(limit=2 ** size // 4)

print(f'group of {size} cells, {2 ** size} assignments; the partial walk sees the first quarter')
print()
print(f"{'cell':>10}  {'honest':>8}  {'truncated':>10}")
for local, cell in enumerate(cells):
    honest = sum(full_cell[local]) / max(sum(full_ways), 1)
    part = sum(part_cell[local]) / max(sum(part_ways), 1)
    flag = '   <- reads as PROVEN SAFE' if part == 0 and honest > 0 else ''
    print(f'{str(hidden[free[cell]]):>10}  {honest:8.1%}  {part:10.1%}{flag}')

group of 5 cells, 32 assignments; the partial walk sees the first quarter

      cell    honest   truncated
    (2, 0)     40.0%        0.0%   <- reads as PROVEN SAFE
    (4, 0)     40.0%        0.0%   <- reads as PROVEN SAFE
    (2, 1)     40.0%       66.7%
    (3, 1)     40.0%       66.7%
    (4, 1)     40.0%       66.7%


### The invariant that follows

> **Never treat 0.0 or 1.0 as merely a small or large number.**

The solver decides those two values from **integer layout counts**, never from the
computed ratio:

```rust
let never_a_mine = cell_ways.iter().all(|&ways| ways == 0.0);
let always_a_mine = cell_ways.iter().zip(&solution.ways)
    .all(|(&mine, &total)| mine == total);
```

A rescaled weight can underflow; a count cannot lie that way. Everything else is
clamped strictly inside `(0, 1)`, so no arithmetic accident can manufacture a proof.

And a **sampled** 0% means only that no draw happened to put a mine there. Monte Carlo
is the fallback for when the exact search gives up, and nothing ever auto-opens on its
numbers.

In [11]:
# The guard, in the Python twin: certainty is read off the integer counts,
# never off the ratio. Note that the search proves cells propagation missed.
for n, ((cells, _), (ways, cell_ways)) in enumerate(zip(components, solutions)):
    for local, cell in enumerate(cells):
        never = all(w == 0 for w in cell_ways[local])
        always = all(m == t for m, t in zip(cell_ways[local], ways))
        verdict = 'PROVEN SAFE' if never else 'PROVEN MINE' if always else 'uncertain'
        print(f'group {n}  {str(hidden[free[cell]]):>8}  {verdict}')

group 0    (2, 0)  uncertain
group 0    (4, 0)  uncertain
group 0    (2, 1)  uncertain
group 0    (3, 1)  uncertain
group 0    (4, 1)  uncertain
group 1    (5, 0)  uncertain
group 1    (5, 1)  uncertain
group 1    (5, 2)  PROVEN SAFE
group 2    (6, 5)  PROVEN SAFE
group 2    (7, 5)  uncertain
group 2    (8, 5)  uncertain


## Where this lives

| What | Where |
|---|---|
| Constraints, propagation, Monte Carlo | `minesweeper_core/src/probability/monte_carlo.rs` |
| Decomposition, combination, caching | `minesweeper_core/src/probability/components.rs` |
| The depth-first search and its budget | `minesweeper_core/src/probability/constraint_search.rs` |
| `certain_cells` | `minesweeper_core/src/probability/mod.rs` |
| Tests, including a brute-force oracle | `minesweeper_core/tests/probability.rs` |

Bounds, as shipped: the search visits at most 1 000 000 nodes (checked at *every*
node, because a deep subtree that never reaches a leaf produces no leaves to count),
past which it reports nothing and the caller falls back to sampling. Worst measured
solve is about 12 ms on boards up to 200 a side, against 137 seconds before
decomposition.

That fallback is the open problem, and not because the positions are hard. The
requirement is that **every** probability reported is the true one — not only the
0.0 and the 1.0 — and a sampled number that looks identical to an exact one does
not meet it. Measured on 30x30/250: 663 solves, 3 refused by the budget, and each
of those solvable exactly in 86-251 ms with the budget removed. They are a few
times past an arbitrary line, not intractable. See *What is left* in
[`constraint-estimator.md`](constraint-estimator.md).